In [5]:
import os

os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"  # see issue #152

In [6]:
def read_file(fname, unescape_newline=True):
    output = []
    with open(fname) as f:
        for line in f:
            output.append(line.strip())
    if unescape_newline:
        output = [l.replace("\\n", "\n") for l in output]
    return output


def get_instructions(inst_dir, df, lp, context_type, split="dev"):
    source_lang = lp.split("-")[0]
    target_lang = lp.split("-")[1].replace("pt", "pt-br")

    xx_yy_insts = read_file(
        f"{inst_dir}/{context_type}/mt/wmt24_chat_{split}.{source_lang}-{target_lang}/instructions.txt"
    )
    yy_xx_insts = read_file(
        f"{inst_dir}/{context_type}/mt/wmt24_chat_{split}.{target_lang}-{source_lang}/instructions.txt"
    )

    assert len(xx_yy_insts) + len(yy_xx_insts) == len(df)
    instructions = []
    j, k = 0, 0

    for i, row in df.iterrows():
        if row["lp"] == f"{source_lang}-{target_lang}":
            instructions.append(xx_yy_insts[j])
            j += 1
        else:
            # nl-en and it-en are considered here
            instructions.append(yy_xx_insts[k])
            k += 1
    return instructions

In [13]:
import pandas as pd

split = "dev"
lang_pair = "en-pt"
df = pd.read_csv(f"../paper_results/{split}.{lang_pair}.csv")
df["comet_diffs"] = (
    df["greedy-7b-ft-w-context-comet"] - df["greedy-7b-ft-wo-context-comet"]
)
instructions_dir = "../instructions"
context_ints = get_instructions(
    instructions_dir, df, lang_pair, "full_context_empty_sys", split
)
non_context_ints = get_instructions(
    instructions_dir, df, lang_pair, "no_context_empty_sys", split
)
# sort the instructions in the same order as the dataframe
df = df.sort_values("comet_diffs", ascending=False)
old_idx = df.index
context_ints = [context_ints[i] for i in old_idx]
non_context_ints = [non_context_ints[i] for i in old_idx]
df = df.reset_index(drop=True)

In [21]:
i = 4
print(context_ints[i])
print(df.loc[i, "greedy-7b-ft-wo-context"])
print(df.loc[i, "greedy-7b-ft-w-context"])

<|im_start|>system
<|im_end|>
<|im_start|>user
Context: Bom dia
Obrigado por entrar em contato com a Ajuda PRS-ORG, o meu nome é NAME-M.
Para começar, pode-me indicar o seu nome por favor?
Meus PRS-ORGs points que comprei não caiu
Efetuei o pagamento e foi descontado
Mais ainda não caiu no jogo
Hello, nice to meet you.
I will surely try my best to help you with missing PRS-ORG Mobile points.
May I know when the purchase was made?
Então?
Hoje
Thanks for the details.
There's nothing to be worried about.

Translate the Brazilian Portuguese source text to English, given the context.
Brazilian Portuguese: Tou vendo
English: <|im_end|>
<|im_start|>assistant

I'm watching.
I see


In [4]:
model_name = "Unbabel/TowerInstruct-v0.2-w-chat-mt-data"

In [5]:
context_outs = df["greedy-7b-ft-w-context"].to_list()
non_context_outs = df["greedy-7b-ft-wo-context"].to_list()

In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer

os.environ["CUDA_VISIBLE_DEVICES"] = "6"
# The model is loaded in 8-bit on available GPUs
model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype="bfloat16", device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [7]:
import inseq

attrib_model = inseq.load_model(model, "saliency")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [48]:
ind = 43
# maybe: 2, 4, 5, 8 (squad building challenge), 14, 20 (saque), maybe 32, 43 (!)
# on 29
out = attrib_model.attribute(
    context_ints[ind],
    context_ints[ind] + " " + context_outs[ind],
    attributed_fn="contrast_prob_diff",
    contrast_targets=context_ints[ind] + " " + non_context_outs[ind],
    step_scores=["contrast_prob_diff"],
    contrast_force_inputs=True,
    show_progress=True,
    generation_args={"max_new_tokens": 30},
    skip_special_tokens=False,
)

Unused arguments during attribution: {'skip_special_tokens': False}
Generation arguments {'max_new_tokens': 30} are provided, but will be ignored (constrained decoding).
Attributing with saliency...: 100%|██████████| 165/165 [00:01<00:00,  3.57it/s]


In [49]:
def is_sentence_ending(tok: str):
    return tok.endswith("<0x0A>") or tok.endswith(":") or tok.endswith(".")


# print(context_ints[ind])
source = context_ints[ind].split("given the context.\n")[-1]
print(f"{source}")
print(f"No context gen: {non_context_outs[ind]}")
print(f"Context gen: {context_outs[ind]}")

start_pos = out[0].attr_pos_start

ends = [
    i + 1
    for i, t in enumerate(out[0].target)
    if is_sentence_ending(t.token) and i < start_pos - 1
] + [start_pos - 1]
starts = [0] + [
    i + 1
    for i, t in enumerate(out[0].target)
    if is_sentence_ending(t.token) and i < start_pos - 1
]

spans = list(zip(starts, ends))
spans = [(s, e) for s, e in spans if e - s > 1]

# out.weight_attributions("contrast_prob_diff")
out = out.aggregate("spans", target_spans=spans)
out.show()

Brazilian Portuguese: Anexos**
English: <|im_end|>
<|im_start|>assistant

No context gen: Annexes**
Context gen: Attachments**


,<0x0A> → ▁Att,▁Anne → ach,xes → ments,**
<s><|im_start|>▁system<0x0A>,0.046,0.072,0.084,0.075
<|im_end|>▁<0x0A>,0.027,0.034,0.028,0.067
<|im_start|>▁user<0x0A>,0.024,0.026,0.029,0.028
Context:,0.018,0.025,0.019,0.019
"▁Obrigado▁por▁entrar▁em▁contato▁com▁a▁Ajuda▁PRS-ORG,▁o▁meu▁nome▁é▁NAME-M.",0.017,0.031,0.015,0.025
<0x0A>,0.009,0.028,0.012,0.022
"Para▁começar,▁pode-me▁indicar▁o▁seu▁nome▁por▁favor?<0x0A>",0.017,0.02,0.012,0.024
Olá▁NAME-M!<0x0A>,0.013,0.014,0.011,0.012
Meu▁nome▁é▁NAME-M.,0.006,0.008,0.005,0.008
<0x0A>,0.008,0.017,0.011,0.009


In [50]:
non_mentions = [
    0.037,
    0.012,
    0.009,
    0.007,
    0.012,
    0.008,
    0.01,
    0.005,
    0.007,
    0.028,
    0.01,
    0.008,
    0.009,
    0.007,
]

mentions_of_dme = [0.021, 0.037, 0.061, 0.014]

print(f"non mentions: {sum(non_mentions) / len(non_mentions):.4f}")
print(f"mentions of dme: {sum(mentions_of_dme) / len(mentions_of_dme):.4f}")

non mentions: 0.0121
mentions of dme: 0.0333


In [51]:
non_send = [
    0.017 + 0.031 + 0.015 + 0.025,
    0.009 + 0.028 + 0.012 + 0.022,
    0.017 + 0.02 + 0.012 + 0.024,
    0.013 + 0.014 + 0.011 + 0.012,
    0.006 + 0.008 + 0.005 + 0.008,
    0.008 + 0.017 + 0.011 + 0.009,
    0.007 + 0.012 + 0.007 + 0.011,
    0.011 + 0.012 + 0.011 + 0.01,
    0.007 + 0.008 + 0.009 + 0.008,
    0.015 + 0.013 + 0.019 + 0.011,
    0.011 + 0.016 + 0.008 + 0.016,
    0.016 + 0.016 + 0.014 + 0.015,
    0.026 + 0.034 + 0.028 + 0.023,
]

sent = [
    0.024 + 0.039 + 0.048 + 0.025,
]

print(f"non send: {sum(non_send) / len(non_send):.4f}")
print(f"sent: {sum(sent) / len(sent):.4f}")

non send: 0.0575
sent: 0.1360
